# Testing `pw_input.py` and `ph_input.py`

This notebook tests the object-oriented QE input layer:

| Module | Classes |
|---|---|
| `pw_input.py` | `ControlNamelist`, `SystemNamelist`, `ElectronsNamelist`, `IonsNamelist`, `CellNamelist`, `AtomicSpeciesCard`, `AtomicPositionsCard`, `KPointsAutoCard`, `CellParametersCard`, `PWInput`, `pw_input_from_atoms` |
| `ph_input.py` | `PhInputph`, `Q2rInput`, `MatdynInput`, `QPointPath`, `PhononWorkflow` |

All tests run **without QE installed** — they only check that the Python objects
build correctly and render valid Fortran input syntax.  
The final section runs the actual calculations if `qe_env` is available.

---
**Structures used:** Si (FCC/diamond), Al (FCC metal), MgO (rock-salt)

---
## ⚙️ Step 1 — Configuration

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION — edit this cell only
# ══════════════════════════════════════════════════════════════════════════════
ENV_ARCHIVE = '/content/drive/MyDrive/conda_envs/qe_env.tar.gz'
# ENV_ARCHIVE = '/content/drive/Shareddrives/QE_Tutorials/qe_env.tar.gz'
# GDRIVE_FILE_ID = 'FILE_ID_HERE'
# ENV_ARCHIVE    = '/content/qe_env.tar.gz'
# ══════════════════════════════════════════════════════════════════════════════
print(f'ENV_ARCHIVE set to: {ENV_ARCHIVE}')

## 🐍 Step 2 — Bootstrap condacolab
> After the kernel restarts, **re-run this cell once**.

In [ ]:
try:
    import condacolab
    condacolab.check()
    print('✅ condacolab active')
except Exception:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'condacolab'],
                          stdout=subprocess.DEVNULL)
    import condacolab
    condacolab.install()

## 📦 Step 3 — Restore `qe_env` and install packages

In [ ]:
import subprocess, os, sys, glob

ENV_PATH = '/usr/local/envs/qe_env'

if 'GDRIVE_FILE_ID' in dir() and not os.path.isfile(ENV_ARCHIVE):
    subprocess.check_call(['pip', 'install', '-q', 'gdown'], stdout=subprocess.DEVNULL)
    import gdown
    gdown.download(id=GDRIVE_FILE_ID, output=ENV_ARCHIVE, quiet=False)
elif ENV_ARCHIVE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

if not os.path.isdir(ENV_PATH):
    print(f'Restoring qe_env …')
    os.makedirs(ENV_PATH, exist_ok=True)
    subprocess.run(['tar', '-xzf', ENV_ARCHIVE, '-C', ENV_PATH], check=True)
    print('✅ Environment restored.')
else:
    print('✅ qe_env already present.')

for sp in glob.glob('/usr/local/envs/qe_env/lib/python*/site-packages'):
    if sp not in sys.path:
        sys.path.insert(0, sp)
        print(f'✅ Added: {sp}')

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ovito'],
                      stdout=subprocess.DEVNULL)

print('\nPackages:')
for pkg in ['numpy', 'matplotlib', 'ase']:
    try:
        __import__(pkg); print(f'  ✅  {pkg}')
    except ImportError as e:
        print(f'  ❌  {pkg} — {e}')

print('\nQE executables:')
for exe in ['pw.x', 'ph.x', 'q2r.x', 'matdyn.x']:
    r = subprocess.run(['conda','run','-n','qe_env','which',exe],
                       capture_output=True, text=True)
    print(f'  {"✅" if r.returncode==0 else "❌"}  {exe}')

## 📥 Step 4 — Download modules and pseudopotentials

In [ ]:
import subprocess, os

# ── Namelist modules — adjust BASE_URL to wherever you host them ─────────────
BASE_URL = 'https://raw.githubusercontent.com/YOUR_USER/YOUR_REPO/main/'

for fname in ['pw_namelists.py', 'ph_namelists.py',
              'postproc_namelists.py', 'pdos_dos_namelists.py',
              'pw_input.py', 'ph_input.py']:
    if not os.path.isfile(fname):
        subprocess.run(['wget', '-q', BASE_URL + fname], check=True)
        print(f'✅ Downloaded {fname}')
    else:
        print(f'✅ {fname} already present')

# ── Pseudopotentials ──────────────────────────────────────────────────────────
PSEUDO_DIR = '/content/pseudo'
os.makedirs(PSEUDO_DIR, exist_ok=True)

PSEUDOS = {
    'Si.pbe-n-kjpaw_psl.1.0.0.UPF':
        'https://pseudopotentials.quantum-espresso.org/upf_files/Si.pbe-n-kjpaw_psl.1.0.0.UPF',
    'Al.pbe-n-kjpaw_psl.1.0.0.UPF':
        'https://pseudopotentials.quantum-espresso.org/upf_files/Al.pbe-n-kjpaw_psl.1.0.0.UPF',
    'Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF':
        'https://pseudopotentials.quantum-espresso.org/upf_files/Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF',
    'O.pbe-n-kjpaw_psl.1.0.0.UPF':
        'https://pseudopotentials.quantum-espresso.org/upf_files/O.pbe-n-kjpaw_psl.1.0.0.UPF',
}

for fname, url in PSEUDOS.items():
    dest = os.path.join(PSEUDO_DIR, fname)
    if not os.path.isfile(dest):
        subprocess.run(['wget', '-q', '-O', dest, url], check=True)
        print(f'✅ {fname}')
    else:
        print(f'✅ {fname} (cached)')

## 📦 Imports and test structures

In [ ]:
from pw_input import (
    ControlNamelist, SystemNamelist, ElectronsNamelist,
    IonsNamelist, CellNamelist,
    AtomicSpeciesCard, AtomicPositionsCard,
    KPointsAutoCard, CellParametersCard,
    PWInput, pw_input_from_atoms,
    ibrav_info, ibrav_celldm_params,
)
from ph_input import (
    PhInputph, Q2rInput, MatdynInput, QPointPath, PhononWorkflow,
)
from ase.build import bulk
from ase import Atoms
import numpy as np

# ── Reference structures ──────────────────────────────────────────────────────
si  = bulk('Si',  'diamond',  a=5.43)   # ibrav=2, FCC
al  = bulk('Al',  'fcc',      a=4.05)   # ibrav=2, FCC metal
mgo = bulk('MgO', 'rocksalt', a=4.21)   # ibrav=2, two-species FCC

# Hexagonal BN monolayer (ibrav=4)
a_bn = 2.504
bn = Atoms(
    'BN',
    positions=[[0, 0, 0],
               [a_bn * 2/3, 0, 0]],
    cell=[[a_bn, 0, 0],
          [-a_bn/2, a_bn*np.sqrt(3)/2, 0],
          [0, 0, 20.0]],    # large vacuum along z
    pbc=True,
)

print('Structures ready')
for name, atoms in [('Si', si), ('Al', al), ('MgO', mgo), ('h-BN', bn)]:
    syms = atoms.get_chemical_symbols()
    print(f'  {name:5s}  nat={len(atoms)}  ntyp={len(set(syms))}  '
          f'a={atoms.cell.lengths()[0]:.3f} Å')

---
# Part 1 — `pw_input.py`

## 1.1 — ibrav metadata

In [ ]:
# Show ibrav info for a few common lattices
for ibrav in [1, 2, 4, 5, 8, 14]:
    info = ibrav_info(ibrav)
    needed = [f'celldm_{i}' for i in info['celldm']]
    print(f"ibrav={ibrav:3d}  {info['label']:<40s}  needs: {needed}")

## 1.2 — `SystemNamelist` ibrav-aware construction

In [ ]:
# ── FCC: only celldm_1 accepted ───────────────────────────────────────────────
s_fcc = SystemNamelist(ibrav=2, ecutwfc=40.0, celldm_1=10.26)
print('FCC system:')
print(s_fcc.to_string())
print()
print('celldm_info():')
print(s_fcc.celldm_info())

In [ ]:
# ── Hexagonal: celldm_1 (a) and celldm_3 (c/a) required ──────────────────────
s_hex = SystemNamelist(ibrav=4, ecutwfc=60.0, celldm_1=4.732, celldm_3=1.631)
print('Hexagonal system:')
print(s_hex.to_string())
print()
print(s_hex.celldm_info())

In [ ]:
# ── Incompatible celldm raises TypeError ─────────────────────────────────────
print('Testing validation...')
tests = [
    (2,  'celldm_3', 1.5,   'c/a not valid for FCC'),
    (2,  'celldm_2', 1.0,   'b/a not valid for FCC'),
    (4,  'celldm_4', 0.5,   'cosine angle not valid for hexagonal'),
    (1,  'celldm_2', 1.0,   'b/a not valid for simple cubic'),
]
for ibrav, bad_key, val, reason in tests:
    try:
        SystemNamelist(ibrav=ibrav, **{bad_key: val})
        print(f'  ❌  ibrav={ibrav} {bad_key}: should have raised!')
    except TypeError:
        print(f'  ✅  ibrav={ibrav} correctly rejects {bad_key}  ({reason})')

## 1.3 — `SystemNamelist.from_atoms()` — extracting celldm from ASE

In [ ]:
# Silicon FCC — celldm(1) auto-extracted in bohr
s_si = SystemNamelist.from_atoms(si, ibrav=2, ecutwfc=40.0, ecutrho=320.0)
print('Si from ASE:')
print(s_si.to_string())
print(f'  (a = {si.cell.lengths()[0]:.4f} Å = {s_si.get("celldm")[0]:.4f} bohr)')

In [ ]:
# Al FCC metal — add smearing parameters
s_al = SystemNamelist.from_atoms(
    al, ibrav=2, ecutwfc=30.0,
    occupations='smearing', smearing='methfessel-paxton', degauss=0.02
)
print('Al from ASE:')
print(s_al.to_string())

In [ ]:
# MgO rock-salt — two species
s_mgo = SystemNamelist.from_atoms(mgo, ibrav=2, ecutwfc=60.0, ecutrho=480.0)
print('MgO from ASE:')
print(s_mgo.to_string())

In [ ]:
# h-BN hexagonal slab — celldm(1) and celldm(3) auto-extracted
s_bn = SystemNamelist.from_atoms(bn, ibrav=4, ecutwfc=60.0)
print('h-BN from ASE (ibrav=4):')
print(s_bn.to_string())
print(f'  c/a = {s_bn.get("celldm")[2]:.4f}  '
      f'(a={bn.cell.lengths()[0]:.3f} Å, c={bn.cell.lengths()[2]:.3f} Å)')

In [ ]:
# ibrav=0 free cell — no celldm set
s_free = SystemNamelist.from_atoms(si, ibrav=0, ecutwfc=40.0)
print('Si ibrav=0 (free cell):')
print(s_free.to_string())
print('  → no celldm; CELL_PARAMETERS card required')

## 1.4 — `KPointsAutoCard` — ibrav-aware mesh constructor

In [ ]:
# Explore accepted arguments for different ibrav values
for ibrav in [1, 2, 4, 5, 6, 8, 12, 14]:
    print(KPointsAutoCard.info(ibrav))
    print()

In [ ]:
# Build meshes for each geometry
meshes = [
    ('Cubic sc      ibrav=1',  KPointsAutoCard(1, nk=8)),
    ('Cubic FCC     ibrav=2',  KPointsAutoCard(2, nk=8)),
    ('Cubic BCC     ibrav=3',  KPointsAutoCard(3, nk=8)),
    ('Hexagonal     ibrav=4',  KPointsAutoCard(4, nk1=8, nk3=6)),
    ('Rhombohedral  ibrav=5',  KPointsAutoCard(5, nk=8)),
    ('Tetragonal    ibrav=6',  KPointsAutoCard(6, nk1=8, nk3=10)),
    ('Orthorhombic  ibrav=8',  KPointsAutoCard(8, nk1=6, nk2=8, nk3=4)),
    ('Monoclinic    ibrav=12', KPointsAutoCard(12, nk1=6, nk2=8, nk3=4)),
    ('Triclinic     ibrav=14', KPointsAutoCard(14, nk1=4, nk2=4, nk3=4)),
    ('FCC shifted   ibrav=2',  KPointsAutoCard(2, nk=8, sk1=1, sk2=1, sk3=1)),
]

for label, k in meshes:
    print(f'{label}:\n  {k.to_string().splitlines()[1].strip()}  → {k!r}')

In [ ]:
# Wrong arguments raise TypeError with a helpful message
error_cases = [
    (2, {'nk1': 8, 'nk2': 8, 'nk3': 8}, 'FCC expects nk, not nk1/2/3'),
    (4, {'nk': 8},                        'Hex expects nk1+nk3, not nk'),
    (8, {'nk1': 6, 'nk2': 8},             'Ortho needs all three nk1,nk2,nk3'),
]
for ibrav, kwargs, reason in error_cases:
    try:
        KPointsAutoCard(ibrav, **kwargs)
        print(f'  ❌  should have raised ({reason})')
    except TypeError as e:
        print(f'  ✅  ibrav={ibrav}: {reason}')
        print(f'       {e}')

## 1.5 — `AtomicSpeciesCard` and `AtomicPositionsCard` from ASE

In [ ]:
# Species card — masses from ASE data, pseudo filename provided
si_pseudos  = {'Si': 'Si.pbe-n-kjpaw_psl.1.0.0.UPF'}
mgo_pseudos = {'Mg': 'Mg.pbe-spnl-kjpaw_psl.1.0.0.UPF',
               'O':  'O.pbe-n-kjpaw_psl.1.0.0.UPF'}

print('ATOMIC_SPECIES for Si:')
print(AtomicSpeciesCard.from_atoms(si, si_pseudos).to_string())
print()
print('ATOMIC_SPECIES for MgO:')
print(AtomicSpeciesCard.from_atoms(mgo, mgo_pseudos).to_string())

In [ ]:
# Positions card — crystal (fractional) coordinates
print('ATOMIC_POSITIONS {crystal} for Si:')
print(AtomicPositionsCard.from_atoms(si, units='crystal').to_string())
print()
print('ATOMIC_POSITIONS {angstrom} for MgO:')
print(AtomicPositionsCard.from_atoms(mgo, units='angstrom').to_string())

In [ ]:
# Positions with fixed constraints (e.g. surface slab: fix bottom layer)
constraints = {0: (0, 0, 0), 1: (0, 0, 0)}   # fix both atoms of Si (demo)
pos_fixed = AtomicPositionsCard.from_atoms(si, units='crystal',
                                           constraints=constraints)
print('ATOMIC_POSITIONS with constraints:')
print(pos_fixed.to_string())

## 1.6 — `CellParametersCard` for ibrav=0

In [ ]:
cell_card = CellParametersCard.from_atoms(si, units='angstrom')
print(cell_card.to_string())
print()
# Or manually:
cell_manual = CellParametersCard('bohr',
    [-2.5685, 0, 2.5685],
    [0, 2.5685, 2.5685],
    [-2.5685, 2.5685, 0])
print(cell_manual.to_string())

## 1.7 — `PWInput` — assembling a complete input file

In [ ]:
# ── Silicon SCF ───────────────────────────────────────────────────────────────
si_inp = PWInput(
    control = ControlNamelist(
        calculation='scf',
        prefix='silicon',
        pseudo_dir='/content/pseudo',
        outdir='/content/out',
        tprnfor=True,
        tstress=True,
    ),
    system = SystemNamelist.from_atoms(si, ibrav=2,
                                       ecutwfc=40.0, ecutrho=320.0),
    electrons = ElectronsNamelist(conv_thr=1e-8),
    atomic_species   = AtomicSpeciesCard.from_atoms(si, si_pseudos),
    atomic_positions = AtomicPositionsCard.from_atoms(si, units='crystal'),
    k_points = KPointsAutoCard(2, nk=8),
)

print(si_inp.to_string())

In [ ]:
# ── Al metallic SCF (smearing) ────────────────────────────────────────────────
al_pseudos = {'Al': 'Al.pbe-n-kjpaw_psl.1.0.0.UPF'}

al_inp = PWInput(
    control = ControlNamelist(
        calculation='scf',
        prefix='aluminium',
        pseudo_dir='/content/pseudo',
        outdir='/content/out',
        tprnfor=True,
    ),
    system = SystemNamelist.from_atoms(
        al, ibrav=2, ecutwfc=30.0, ecutrho=240.0,
        occupations='smearing',
        smearing='methfessel-paxton',
        degauss=0.02,
    ),
    electrons = ElectronsNamelist(conv_thr=1e-8, mixing_beta=0.5),
    atomic_species   = AtomicSpeciesCard.from_atoms(al, al_pseudos),
    atomic_positions = AtomicPositionsCard.from_atoms(al, units='crystal'),
    k_points = KPointsAutoCard(2, nk=12, sk1=1, sk2=1, sk3=1),
)

print(al_inp.to_string())

In [ ]:
# ── MgO two-species rock-salt ─────────────────────────────────────────────────
mgo_inp = PWInput(
    control = ControlNamelist(
        calculation='scf',
        prefix='mgo',
        pseudo_dir='/content/pseudo',
        outdir='/content/out',
    ),
    system = SystemNamelist.from_atoms(mgo, ibrav=2,
                                       ecutwfc=60.0, ecutrho=480.0),
    electrons = ElectronsNamelist(conv_thr=1e-9),
    atomic_species   = AtomicSpeciesCard.from_atoms(mgo, mgo_pseudos),
    atomic_positions = AtomicPositionsCard.from_atoms(mgo, units='crystal'),
    k_points = KPointsAutoCard(2, nk=6),
)

print(mgo_inp.to_string())

In [ ]:
# ── ibrav=0 free cell (no celldm) ────────────────────────────────────────────
si_free_inp = PWInput(
    control = ControlNamelist(calculation='scf', prefix='si_free',
                              pseudo_dir='/content/pseudo', outdir='/content/out'),
    system  = SystemNamelist.from_atoms(si, ibrav=0, ecutwfc=40.0),
    electrons = ElectronsNamelist(),
    atomic_species   = AtomicSpeciesCard.from_atoms(si, si_pseudos),
    atomic_positions = AtomicPositionsCard.from_atoms(si, units='crystal'),
    k_points         = KPointsAutoCard(0, nk1=8, nk2=8, nk3=8),
    cell_parameters  = CellParametersCard.from_atoms(si, units='angstrom'),
)

print(si_free_inp.to_string())

## 1.8 — `pw_input_from_atoms()` one-shot factory

In [ ]:
# Minimal one-liner for Si SCF
si_quick = pw_input_from_atoms(
    si,
    ibrav=2,
    ecutwfc=40.0,
    pseudos=si_pseudos,
    prefix='silicon',
    pseudo_dir='/content/pseudo',
    outdir='/content/out',
    k_points=KPointsAutoCard(2, nk=8),
)
print(si_quick)
print()
print(si_quick.to_string())

In [ ]:
# Write to file and verify it looks reasonable
import os
os.makedirs('/content/out', exist_ok=True)
si_quick.write('/content/si_scf.in')

# Read it back and print
with open('/content/si_scf.in') as f:
    print(f.read())

---
# Part 2 — `ph_input.py`

## 2.1 — `PhInputph.single_q()` — Gamma-point phonons + dielectric

In [ ]:
ph_gamma = (
    PhInputph.single_q(
        prefix='silicon',
        qpoint=(0, 0, 0),
        fildyn='/content/ph/si.dynG',
        outdir='/content/out',
    )
    .with_dielectric()    # epsil + zeu
)
print(ph_gamma)
print()
print(ph_gamma.to_string())

In [ ]:
# With Raman tensor (auto-enables dielectric)
ph_raman = (
    PhInputph.single_q('silicon', (0, 0, 0),
                        fildyn='/content/ph/si.dynG',
                        outdir='/content/out')
    .with_raman(elop=True)
)
print(ph_raman.to_string())
print()
# Check that epsil was auto-enabled
assert ph_raman._params['epsil'] == True,  'epsil should be True'
assert ph_raman._params['lraman'] == True, 'lraman should be True'
assert ph_raman._params['elop'] == True,   'elop should be True'
print('✅ Raman auto-enables dielectric')

## 2.2 — `PhInputph.dispersion()` — uniform q-grid

In [ ]:
ph_disp = (
    PhInputph.dispersion(
        prefix='silicon',
        nq1=4, nq2=4, nq3=4,
        fildyn='/content/ph/si.dyn',
        outdir='/content/out',
    )
    .with_dielectric()
)
print(ph_disp.to_string())

In [ ]:
# With recover for restart
ph_disp_restart = (
    PhInputph.dispersion('silicon', 4, 4, 4,
                          fildyn='/content/ph/si.dyn',
                          outdir='/content/out')
    .with_dielectric()
    .with_recover()
)
print(ph_disp_restart.to_string())

In [ ]:
# Splitting a run: q-points 1-4, irreps 1-2 only
ph_split = (
    PhInputph.dispersion('silicon', 4, 4, 4,
                          fildyn='/content/ph/si.dyn',
                          outdir='/content/out')
    .with_dielectric()
    .parallel(start_q=1, last_q=4, start_irr=1, last_irr=2)
)
print(ph_split.to_string())

## 2.3 — `PhInputph.dispersion()` + electron-phonon

In [ ]:
# Lambda + phonon linewidths for Al (metal)
ph_eph = (
    PhInputph.dispersion(
        prefix='aluminium',
        nq1=4, nq2=4, nq3=4,
        fildyn='/content/ph/al.dyn',
        outdir='/content/out',
    )
    .with_eph(
        method='lambda_tetra',
        el_ph_sigma=0.02,
        nk1=16, nk2=16, nk3=16,
        fildvscf='/content/ph/al.dvscf',
    )
)
print(ph_eph.to_string())

In [ ]:
# AHC self-energy workflow
ph_ahc = (
    PhInputph.dispersion('silicon', 4, 4, 4,
                          fildyn='/content/ph/si.dyn',
                          outdir='/content/out')
    .with_ahc(
        ahc_dir='/content/ahc',
        ahc_nbnd=20,
        ahc_nbndskip=0,
        skip_upperfan=False,
    )
)
print(ph_ahc.to_string())

## 2.4 — `PhInputph.qplot()` — explicit q-point list

In [ ]:
# Band-path form in crystal coordinates (FCC high-symmetry path)
# Γ → X → W → K → Γ → L
qpath_fcc = [
    (0.000, 0.000, 0.000, 10),   # Γ
    (0.500, 0.000, 0.500, 10),   # X
    (0.500, 0.250, 0.750, 10),   # W
    (0.375, 0.375, 0.750, 10),   # K
    (0.000, 0.000, 0.000, 10),   # Γ
    (0.000, 0.500, 0.500, 1),    # L  ← last point: npoints=1
]

ph_qplot = PhInputph.qplot(
    prefix='silicon',
    qpoints=qpath_fcc,
    fildyn='/content/ph/si.dynpath',
    outdir='/content/out',
    q_in_cryst_coord=True,
    q_in_band_form=True,
)
print(ph_qplot.to_string())

## 2.5 — `Q2rInput`

In [ ]:
q2r = Q2rInput(
    fildyn='/content/ph/si.dyn',
    flfrc='/content/ph/si.fc',
    zasr='crystal',
)
print(q2r)
print()
print(q2r.to_string())

In [ ]:
# Invalid zasr raises ValueError
try:
    Q2rInput(fildyn='x.dyn', flfrc='x.fc', zasr='wrong')
    print('❌ should have raised')
except ValueError as e:
    print(f'✅ Invalid zasr correctly rejected: {e}')

## 2.6 — `QPointPath` — band paths for `matdyn.x`

In [ ]:
# FCC high-symmetry path: Γ-X-W-K-Γ-L
fcc_path = (
    QPointPath(cryst=True, band_form=True)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.500, 0.000, 0.500, name='X', npoints=40)
    .add(0.500, 0.250, 0.750, name='W', npoints=40)
    .add(0.375, 0.375, 0.750, name='K', npoints=40)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.000, 0.500, 0.500, name='L', npoints=1)
)
print(f'Path has {len(fcc_path)} points')
print(fcc_path.to_string())

In [ ]:
# Hexagonal path: Γ-M-K-Γ-A
hex_path = (
    QPointPath(cryst=True, band_form=True)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.500, 0.000, 0.000, name='M', npoints=40)
    .add(1/3,   1/3,   0.000, name='K', npoints=40)
    .add(0.000, 0.000, 0.000, name='Γ', npoints=40)
    .add(0.000, 0.000, 0.500, name='A', npoints=1)
)
print(hex_path.to_string())

## 2.7 — `MatdynInput` — dispersion and DOS

In [ ]:
# Phonon dispersion along the FCC path
md_disp = MatdynInput.dispersion(
    flfrc='/content/ph/si.fc',
    qpath=fcc_path,
    asr='crystal',
    flfrq='/content/ph/si.freq',
    flvec='/content/ph/si.modes',
)
print(f'mode = {md_disp.mode}')
print()
print(md_disp.to_string())

In [ ]:
# Phonon DOS on a dense mesh
md_dos = MatdynInput.dos(
    flfrc='/content/ph/si.fc',
    nk1=16, nk2=16, nk3=16,
    asr='crystal',
    fldos='/content/ph/si.dos',
    deltaE=2.0,
    degauss=5.0,
)
print(f'mode = {md_dos.mode}')
print()
print(md_dos.to_string())

In [ ]:
# Mode check: MatdynInput.dos() passed to matdyn_disp should raise
try:
    PhononWorkflow(ph_disp, q2r, matdyn_disp=md_dos)
    print('❌ should have raised')
except ValueError as e:
    print(f'✅ Correctly rejected: {e}')

## 2.8 — `PhononWorkflow` — full workflow container

In [ ]:
# Assemble the full Si phonon workflow
wf = PhononWorkflow(
    ph          = ph_disp,
    q2r         = q2r,
    matdyn_disp = md_disp,
    matdyn_dos  = md_dos,
    conda_env   = 'qe_env',
)
print(wf)

In [ ]:
# All input files in one view
print(wf.to_string())

In [ ]:
# Run commands — q2r and matdyn are always serial
print('Serial:')
for cmd in wf.run_commands(nproc=1):
    print(' ', cmd)
print()
print('Parallel (4 MPI tasks for ph.x):')
for cmd in wf.run_commands(nproc=4):
    print(' ', cmd)

In [ ]:
# Write all input files to disk
import os
os.makedirs('/content/ph', exist_ok=True)
wf.write('/content/ph')

print()
for f in sorted(os.listdir('/content/ph')):
    print(f'  {f}')

In [ ]:
# fildyn mismatch warning
import warnings
q2r_wrong = Q2rInput(fildyn='wrong.dyn', flfrc='si.fc')
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    PhononWorkflow(ph_disp, q2r_wrong)
    if w:
        print(f'✅ Mismatch warning fired: {w[0].message}')
    else:
        print('❌ No warning fired')

---
# Part 3 — Full workflow: run with QE

These cells actually execute the calculations.  
They will be **skipped silently** if `qe_env` is not available.

In [ ]:
import subprocess

def qe_available():
    r = subprocess.run(['conda','run','-n','qe_env','which','pw.x'],
                       capture_output=True)
    return r.returncode == 0

QE_OK = qe_available()
print('QE available:', '✅' if QE_OK else '❌ (skipping run cells)')

In [ ]:
if not QE_OK:
    print('Skipping — qe_env not found.')
else:
    # Write the SCF input and run
    si_quick.write('/content/si_scf.in')

    r = subprocess.run(
        ['conda', 'run', '-n', 'qe_env', 'pw.x', '-input', '/content/si_scf.in'],
        capture_output=True, text=True
    )
    print('\n'.join(r.stdout.splitlines()[-30:]))
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])

In [ ]:
if not QE_OK:
    print('Skipping.')
else:
    # Parse total energy from output
    with open('/content/si_scf.in') as f:
        inp_text = f.read()

    r = subprocess.run(
        ['conda', 'run', '-n', 'qe_env', 'pw.x', '-input', '/content/si_scf.in'],
        capture_output=True, text=True
    )

    # Extract total energy
    for line in r.stdout.splitlines():
        if '!    total energy' in line:
            print('Total energy:', line.strip())
        if 'convergence has been achieved' in line:
            print(line.strip())

In [ ]:
if not QE_OK:
    print('Skipping.')
else:
    # Run phonon at Gamma
    ph_gamma_run = (
        PhInputph.single_q(
            prefix='silicon',
            qpoint=(0, 0, 0),
            fildyn='/content/ph/si.dynG',
            outdir='/content/out',
        )
        .with_dielectric()
    )
    os.makedirs('/content/ph', exist_ok=True)
    with open('/content/ph/ph_gamma.in', 'w') as f:
        f.write(ph_gamma_run.to_string())

    r = subprocess.run(
        ['conda', 'run', '-n', 'qe_env', 'ph.x', '-input', '/content/ph/ph_gamma.in'],
        capture_output=True, text=True
    )
    print('\n'.join(r.stdout.splitlines()[-30:]))
    if r.returncode != 0:
        print('STDERR:', r.stderr[-1000:])